In [1]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
CODE_DIR = os.environ.get('AGES_CODE', '/content/drive/MyDrive/ages-resolution-time/code')
sys.path.insert(0, CODE_DIR); os.chdir(CODE_DIR)

sys.argv = ['notebook'] + []


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import numpy as np
import pandas as pd
import joblib
import ast
import time

from ages_common import (RANDOM_STATE, HORIZON_DAYS, SENSITIVITY_HORIZONS, TARGET,
                         RAW_CSV, DIR_PREP, ensure, save_versions)

np.random.seed(RANDOM_STATE)
OUT = ensure(DIR_PREP)
print('Output folder:', OUT)
save_versions(f'{OUT}/environment_versions.txt')

['Python==3.13.18',
 'pandas==2.2.3',
 'numpy==2.4.6',
 'sklearn==1.6.1',
 'xgboost==3.4.1',
 'imblearn==0.14.2',
 'scipy==1.16.3',
 'statsmodels==0.15.0',
 'matplotlib==3.10.0']

In [3]:
# Load raw data
df = pd.read_csv(RAW_CSV)
print(f'Raw: {len(df)} rows, {len(df.columns)} columns, '
      f'{df["Repository"].nunique()} repositories')
n_repos_raw = df['Repository'].nunique()

df['Created At'] = pd.to_datetime(df['Created At'], utc=True, errors='coerce')
df['Closed At']  = pd.to_datetime(df['Closed At'],  utc=True, errors='coerce')
df['Time_Taken_Days'] = (df['Closed At'] - df['Created At']).dt.total_seconds() / 86400

n_repos_valid = n_repos_raw
n_repos_365 = df['Repository'].nunique()

n0 = len(df)
df = df[df['Time_Taken_Days'] >= 0].dropna(subset=['Time_Taken_Days', 'Created At'])
n1 = len(df)

OUTLIER_DAYS = 365
df = df[df['Time_Taken_Days'] <= OUTLIER_DAYS].reset_index(drop=True)
n2 = len(df)

print(f'Raw:                 {n0}')
print(f'Valid durations:     {n1}  (-{n0-n1})')
print(f'Duration <= {OUTLIER_DAYS}d:      {n2}  (-{n1-n2})')

Raw: 23006 rows, 15 columns, 27 repositories
Raw:                 23006
Valid durations:     23006  (-0)
Duration <= 365d:      22059  (-947)


In [4]:
# Maturity buffer: sensitivity across candidate buffers (45-365 days)
data_max_close = df['Closed At'].max()
rows = []
for B in [0, 45, 90, 180, 270, 365]:
    cut = data_max_close - pd.Timedelta(days=B)
    sub = df[df['Created At'] <= cut].copy()
    q = sub['Created At'].dt.tz_convert(None).dt.to_period('Q')
    lr = (sub['Time_Taken_Days'] > HORIZON_DAYS).astype(int)
    by_q = lr.groupby(q).mean()
    last_q = by_q.index.max()
    hist = lr[q < (last_q - 1)].mean()
    rows.append({'Buffer (days)': B, 'Cutoff': str(cut.date()), 'N retained': len(sub),
                 'Repositories': sub['Repository'].nunique(),
                 'Last quarter': str(last_q),
                 'Long-resolution rate, last quarter': round(by_q.iloc[-1], 3),
                 'Long-resolution rate, earlier quarters': round(hist, 3),
                 'Long-resolution rate, all retained': round(lr.mean(), 3)})
df_buffer = pd.DataFrame(rows)
df_buffer.to_csv(f'{OUT}/table_maturity_buffer_sensitivity.csv', index=False)
print(df_buffer.to_string(index=False))

 Buffer (days)     Cutoff  N retained  Repositories Last quarter  Long-resolution rate, last quarter  Long-resolution rate, earlier quarters  Long-resolution rate, all retained
             0 2024-11-08       22059            27       2024Q4                               0.034                                   0.376                               0.293
            45 2024-09-24       19136            27       2024Q3                               0.179                                   0.389                               0.329
            90 2024-08-10       16268            26       2024Q3                               0.208                                   0.389                               0.358
           180 2024-05-12       13538            22       2024Q2                               0.319                                   0.399                               0.384
           270 2024-02-12       11648            22       2024Q1                               0.301               

In [5]:
# Apply the selected maturity buffer (180 days)
MATURITY_BUFFER_DAYS = 180
cutoff = data_max_close - pd.Timedelta(days=MATURITY_BUFFER_DAYS)
print(f'Latest recorded closure: {data_max_close}')
print(f'Maturity cutoff:         {cutoff}')

n_before = len(df)
df = df[df['Created At'] <= cutoff].reset_index(drop=True)
n_after = len(df)
print(f'Removed {n_before - n_after} issues created after cutoff '
      f'({100*(n_before-n_after)/n_before:.1f}%)')
print(f'Remaining: {n_after}')

tmp = df.copy()
tmp['_q'] = tmp['Created At'].dt.tz_convert(None).dt.to_period('Q').astype(str)
tmp['lr14'] = (tmp['Time_Taken_Days'] > HORIZON_DAYS).astype(int)
quarterly = tmp.groupby('_q')['lr14'].agg(['mean', 'count'])
quarterly.to_csv(f'{OUT}/table_long_resolution_rate_by_quarter.csv')
print('Long-resolution rate by quarter (after maturity filter):')
print(quarterly.tail(12))
del tmp

Latest recorded closure: 2024-11-08 08:56:49+00:00
Maturity cutoff:         2024-05-12 08:56:49+00:00
Removed 8521 issues created after cutoff (38.6%)
Remaining: 13538
Long-resolution rate by quarter (after maturity filter):
            mean  count
_q                     
2021Q3  0.421918    365
2021Q4  0.419192    396
2022Q1  0.393750    480
2022Q2  0.392704    466
2022Q3  0.364606    469
2022Q4  0.457786    533
2023Q1  0.390057    523
2023Q2  0.434343    792
2023Q3  0.346336    846
2023Q4  0.378981    942
2024Q1  0.316888   1581
2024Q2  0.318729    913


In [6]:
# Target: long-resolution (> 14 days) vs short-resolution (<= 14 days)
df[TARGET] = (df['Time_Taken_Days'] > HORIZON_DAYS).astype(int)
print(f'Target ({HORIZON_DAYS}-day horizon): {df[TARGET].mean()*100:.1f}% long-resolution')

# alternative horizons for the sensitivity analysis
for h in SENSITIVITY_HORIZONS:
    df[f'{TARGET}_{h}d'] = (df['Time_Taken_Days'] > h).astype(int)

cleaning_log = pd.DataFrame([
    {'Step': 'Raw dataset', 'N': n0, 'Repositories': n_repos_raw},
    {'Step': 'Valid durations', 'N': n1, 'Repositories': n_repos_valid},
    {'Step': f'Duration <= {OUTLIER_DAYS}d', 'N': n2, 'Repositories': n_repos_365},
    {'Step': f'Maturity filter ({MATURITY_BUFFER_DAYS}d buffer)', 'N': n_after,
     'Repositories': df['Repository'].nunique()},
])
cleaning_log.to_csv(f'{OUT}/table_cleaning_log.csv', index=False)
print(cleaning_log)

Target (14-day horizon): 38.4% long-resolution
                            Step      N  Repositories
0                    Raw dataset  23006            27
1                Valid durations  23006            27
2               Duration <= 365d  22059            27
3  Maturity filter (180d buffer)  13538            22


In [7]:
# Feature engineering (field Comments excluded: not available at creation time)
def parse_list(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', 'nan'):
        return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return [str(val)]

df['n_assignees']      = df['Assignees'].apply(lambda x: len(parse_list(x)))
df['n_labels']         = df['Labels'].apply(lambda x: len(parse_list(x)))
df['is_feature_label'] = df['Labels'].apply(
    lambda x: int(any('feature' in str(l).lower() for l in parse_list(x))))
df['title_len']        = df['Title'].fillna('').apply(len)
df['Contributors Expertise'] = pd.to_numeric(df['Contributors Expertise'], errors='coerce')
df['has_milestone']    = df['Milestone'].notna().astype(int)
df['created_hour']     = df['Created At'].dt.hour
df['created_dayofweek']= df['Created At'].dt.dayofweek

df['body_len']     = df['Body'].fillna('').apply(len)
df['log_body_len'] = np.log1p(df['body_len'])

df['cat_bug']         = (df['Category'] == 'Bug').astype(int)
df['cat_feature']     = (df['Category'] == 'Feature').astype(int)
df['cat_improvement'] = (df['Category'] == 'Improvement').astype(int)

COMPONENTS = ['ui','backend','api','network','security','database','authentication','performance']
for comp in COMPONENTS:
    df[f'comp_{comp}'] = df['Component'].fillna('').str.contains(comp, case=False).astype(int)

print('Base features created')
print(f'has_milestone=1 for {df["has_milestone"].mean()*100:.1f}% of issues')

Base features created
has_milestone=1 for 19.4% of issues


In [8]:
# Keyword-based text features (complexity, urgency, negative words)
complexity_words = ['refactor', 'breaking change', 'breaking', 'redesign', 'rewrite',
                    'architecture', 'migration', 'deprecat', 'incompatible', 'blocked',
                    'depends', 'dependency', 'complex', 'investigat', 'unclear', 'discussion']
urgency_words    = ['urgent', 'critical', 'asap', 'blocker', 'regression', 'security']
neg_words        = ['error','fail','broken','crash','bug','problem','issue','wrong',
                    'cannot', "doesn't", 'not working', 'unexpected']

def count_words(text, words):
    t = str(text).lower()
    return sum(t.count(w) for w in words)

df['n_complexity_words'] = df['Body'].apply(lambda x: count_words(x, complexity_words))
df['n_urgency_words']    = df['Body'].apply(lambda x: count_words(x, urgency_words))
df['n_negative_words']   = df['Body'].apply(lambda x: count_words(x, neg_words))
df['body_has_steps']     = df['Body'].fillna('').str.contains(
    r'step|repro|reproduce|expected|actual', case=False, na=False, regex=True).astype(int)
print('Keyword features created')

Keyword features created


In [9]:
# Assignee history: only issues closed before the creation time of the current issue

df['first_assignee'] = df['Assignees'].apply(lambda x: parse_list(x)[0] if parse_list(x) else 'none')
df = df.sort_values('Created At').reset_index(drop=True)

df['assignee_avg_duration'] = np.nan
df['assignee_task_count']   = 0
df['assignee_open_tasks']   = 0

t0 = time.time()
for a, grup in df.groupby('first_assignee'):
    idxs = grup.index
    cr  = grup['Created At'].values
    cl  = grup['Closed At'].values
    dur = grup['Time_Taken_Days'].values
    for i, idx in enumerate(idxs):
        c = cr[i]
        closed_before = cl < c
        if closed_before.any():
            df.at[idx, 'assignee_avg_duration'] = dur[closed_before].mean()
            df.at[idx, 'assignee_task_count']   = int(closed_before.sum())
        df.at[idx, 'assignee_open_tasks'] = int(((cr < c) & (cl > c)).sum())

global_median = df['Time_Taken_Days'].median()
df['assignee_avg_duration'] = df['assignee_avg_duration'].fillna(global_median)

print(f'Computed in {time.time()-t0:.1f}s')
print(f'Issues with no assignee history (imputed with global median): '
      f'{(df["assignee_task_count"]==0).sum()}')
print(f'Global median resolution time used for imputation: {global_median:.3f} days')

Computed in 1.5s
Issues with no assignee history (imputed with global median): 1427
Global median resolution time used for imputation: 6.938 days


In [10]:
# Missing values
missing = df.isnull().sum()
print('Missing before imputation:')
print(missing[missing > 0])
df['Contributors Expertise'] = df['Contributors Expertise'].fillna(
    df['Contributors Expertise'].median())

Missing before imputation:
Milestone    10917
Body           244
dtype: int64


In [11]:
# Final feature set
FEATURE_COLS = [
    'log_body_len', 'Contributors Expertise',
    'n_labels', 'has_milestone', 'is_feature_label', 'title_len',
    'created_dayofweek', 'created_hour', 'n_assignees',
    'cat_bug', 'cat_feature', 'cat_improvement',
] + [f'comp_{c}' for c in COMPONENTS] + [
    'n_complexity_words', 'n_urgency_words', 'body_has_steps', 'n_negative_words',
    'assignee_avg_duration', 'assignee_task_count', 'assignee_open_tasks',
]
print(f'{len(FEATURE_COLS)} features')
print(f'Missing values in feature set: {df[FEATURE_COLS].isnull().sum().sum()} (must be 0)')

27 features
Missing values in feature set: 0 (must be 0)


In [12]:
# Association between contributor expertise and number of assignees.
from scipy.stats import spearmanr
rho, p = spearmanr(df['Contributors Expertise'], df['n_assignees'])
print(f'Spearman Contributors Expertise vs n_assignees: rho={rho:.3f}, p={p:.2g}')
pd.DataFrame([{'Pair': 'Contributors Expertise vs n_assignees', 'rho': rho, 'p': p}]).to_csv(
    f'{OUT}/table_expertise_assignees_spearman.csv', index=False)

Spearman Contributors Expertise vs n_assignees: rho=-0.053, p=5.2e-10


In [13]:
# Repository and category distribution
bal = pd.DataFrame([{'Horizon (days)': h, 'N': len(df),
                     'Long-resolution (n)': int(df[f'{TARGET}_{h}d'].sum()),
                     'Long-resolution (%)': round(100*df[f'{TARGET}_{h}d'].mean(), 1)}
                    for h in SENSITIVITY_HORIZONS])
bal.to_csv(f'{OUT}/table_horizon_class_balance.csv', index=False)
print(bal)

   Horizon (days)      N  Long-resolution (n)  Long-resolution (%)
0               7  13538                 6715                 49.6
1              14  13538                 5203                 38.4
2              21  13538                 4440                 32.8
3              28  13538                 3897                 28.8


In [14]:
# Save outputs
keep_cols = FEATURE_COLS + [TARGET] + [f'{TARGET}_{h}d' for h in SENSITIVITY_HORIZONS] + [
    'Time_Taken_Days', 'Created At', 'Repository', 'Category', 'Number',
    'first_assignee']
df_final = df[keep_cols].copy()
df_final.to_csv(f'{OUT}/df_final.csv', index=False)

joblib.dump(FEATURE_COLS, f'{OUT}/feature_cols.pkl')
print(f'Saved to {OUT}')
print(f'   df_final.csv: {len(df_final)} rows, {len(df_final.columns)} columns')

Saved to /content/drive/MyDrive/ages-resolution-time/pipeline/01_data_prep
   df_final.csv: 13538 rows, 38 columns


In [15]:
# Median resolution time: full dataset vs training sets (imputation check)
from ages_common import random_split, chronological_split

X_tr, X_te, y_tr, y_te = random_split(df_final, FEATURE_COLS)
*_, d_sorted, k = chronological_split(df_final, FEATURE_COLS)

medians = pd.DataFrame([
    {'Set': 'Full dataset',
     'Median resolution time (days)': df_final['Time_Taken_Days'].median()},
    {'Set': 'Random split, training set',
     'Median resolution time (days)': df_final.loc[X_tr.index, 'Time_Taken_Days'].median()},
    {'Set': 'Chronological split, training set',
     'Median resolution time (days)': d_sorted['Time_Taken_Days'].iloc[:k].median()},
])
medians.to_csv(f'{OUT}/table_median_resolution_time.csv', index=False)
print(medians.round(3))

                                 Set  Median resolution time (days)
0                       Full dataset                          6.938
1         Random split, training set                          6.856
2  Chronological split, training set                          7.421
